# Policy Document Diff & Impact Analysis
## Comparing Old and New Policies, Summarising Business and Process Impact

**Project -- NLP / Enterprise Document Intelligence Portfolio**

---

When organisations update policies -- compliance, HR, security, data
governance -- stakeholders need to understand **what changed** and
**what it means for them**. Reading two 20-page documents side-by-side
is slow, error-prone, and does not scale.

This notebook builds a complete **policy diff and impact analysis** system:

1. **Diff** -- Detect additions, deletions, and modifications at the clause level
2. **Classify** -- Tag each change by type (substantive, editorial, structural)
3. **Summarise** -- Generate structured impact summaries per business function
4. **Report** -- Produce a stakeholder-ready change impact report

The techniques apply to any document versioning scenario: contracts,
regulations, standard operating procedures, and internal guidelines.

## Table of Contents

| Part | Topic | Purpose |
|---|---|---|
| 1 | Document diffing explained | Algorithms and trade-offs |
| 2 | Policy corpus | 6 policy domains, old vs new versions |
| 3 | Clause-level diff engine | Detect changes between document versions |
| 4 | Change classifier | Substantive vs editorial vs structural |
| 5 | Structured summarisation | Impact summary per business function |
| 6 | End-to-end pipeline | Full diff-to-report workflow |
| 7 | Evaluation | Quality metrics |
| 8 | Visualisations | Change analysis dashboards |
| 9 | Production patterns | Real-world deployment |

## Part 1: Document Diffing Explained

### Why Not Just Use `diff`?

Unix `diff` works on lines. But policy documents have semantic structure:
clauses, sub-clauses, definitions, and cross-references. A line-level diff
produces noise (whitespace, reformatting) and misses meaning (a moved clause
looks like a delete + add).

### Three Levels of Document Diffing

| Level | Granularity | Good For | Misses |
|---|---|---|---|
| **Token diff** | Word-by-word | Precise edit location | Semantic meaning |
| **Sentence diff** | Sentence-by-sentence | Clause-level changes | Structural moves |
| **Semantic diff** | Meaning-based | Paraphrased rewrites | Implementation cost |

This notebook uses **clause-level diffing with semantic similarity** --
a hybrid that gets the best of sentence-level precision and semantic
understanding.

### The Algorithm

```
OLD DOCUMENT                    NEW DOCUMENT
  clause_1  ──────────────────>  clause_1'     (MODIFIED: similarity > 0.6)
  clause_2  ──────────────────>  clause_2      (UNCHANGED: similarity > 0.95)
  clause_3  ─── X                              (DELETED: no match > 0.6)
                                 clause_4      (ADDED: no match in old)
  clause_5  ──────────────────>  clause_5'     (MODIFIED: similarity > 0.6)
```

1. Split both documents into clauses
2. Compute pairwise similarity between all old and new clauses
3. Use greedy best-match alignment (similarity threshold)
4. Classify unmatched old clauses as DELETED, unmatched new as ADDED
5. For matched pairs: UNCHANGED if similarity > 0.95, MODIFIED otherwise

### Change Classification

Not all changes matter equally:

| Type | Definition | Example | Business Impact |
|---|---|---|---|
| **Substantive** | Changes meaning, obligations, or rights | SLA target changed from 99.9%% to 99.5%% | HIGH |
| **Editorial** | Rephrasing without meaning change | "shall" -> "will" | LOW |
| **Structural** | Reorganisation, renumbering, moved text | Clause 3.2 moved to 4.1 | NONE |

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ['pandas', 'numpy', 'matplotlib', 'seaborn', 'plotly']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Ready.')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import re, math, json as json_mod, textwrap, hashlib, copy, difflib
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 25)
pd.set_option('display.max_colwidth', 160)
plt.rcParams['figure.figsize'] = (14, 5)
sns.set_style('whitegrid')
np.random.seed(42)
print('Imports OK')

## Part 2: Policy Corpus

Six policy domains, each with an **old** (v1) and **new** (v2) version.
The differences simulate realistic corporate policy updates: tightened
security requirements, expanded compliance scope, updated SLAs, and
new data handling provisions.

In [ ]:
@dataclass
class PolicyClause:
    clause_id: str
    text: str
    section: str


@dataclass
class PolicyDocument:
    name: str
    domain: str
    version: str
    clauses: list

    @property
    def full_text(self):
        return ' '.join(c.text for c in self.clauses)


OLD_POLICIES = {
    'data_security': PolicyDocument(
        'Data Security Policy', 'data_security', 'v1',
        [
            PolicyClause('DS-1.1', 'All data at rest shall be encrypted using AES-256. '
                'Encryption keys are managed by the IT Security team and rotated annually.',
                'Encryption'),
            PolicyClause('DS-1.2', 'Data in transit shall be protected using TLS 1.2 '
                'or higher for all external communications.',
                'Encryption'),
            PolicyClause('DS-2.1', 'Access to production systems requires manager approval '
                'and is reviewed quarterly. All access follows the principle of least privilege.',
                'Access Control'),
            PolicyClause('DS-2.2', 'Multi-factor authentication is required for all '
                'administrative accounts. Standard user accounts may use single-factor '
                'authentication.',
                'Access Control'),
            PolicyClause('DS-3.1', 'Vulnerability scans shall be conducted monthly '
                'on all internet-facing systems. Critical vulnerabilities shall be '
                'remediated within 30 days.',
                'Vulnerability Management'),
            PolicyClause('DS-3.2', 'External penetration testing shall be performed '
                'annually by a qualified third party.',
                'Vulnerability Management'),
            PolicyClause('DS-4.1', 'Security incidents shall be reported to the '
                'IT Security team within 24 hours of detection. A post-incident '
                'review shall be completed within 30 days.',
                'Incident Response'),
        ]),

    'data_governance': PolicyDocument(
        'Data Governance Policy', 'data_governance', 'v1',
        [
            PolicyClause('DG-1.1', 'All data assets shall be classified into four '
                'categories: Public, Internal, Confidential, and Restricted. '
                'Classification is the responsibility of the data owner.',
                'Data Classification'),
            PolicyClause('DG-1.2', 'Data owners shall be identified for all critical '
                'data assets. Ownership is assigned at the department head level.',
                'Data Classification'),
            PolicyClause('DG-2.1', 'Personal data shall be retained for no longer '
                'than necessary for the purpose it was collected. Default retention '
                'period is 3 years.',
                'Retention'),
            PolicyClause('DG-2.2', 'Data deletion requests shall be processed within '
                '30 business days. Confirmation of deletion shall be provided to '
                'the requesting party.',
                'Retention'),
            PolicyClause('DG-3.1', 'Cross-border data transfers require approval from '
                'the Legal department. Transfers to non-adequate countries require '
                'Standard Contractual Clauses.',
                'Cross-Border'),
        ]),

    'acceptable_use': PolicyDocument(
        'Acceptable Use Policy', 'acceptable_use', 'v1',
        [
            PolicyClause('AU-1.1', 'Company IT resources are provided for business '
                'purposes. Limited personal use is permitted provided it does not '
                'interfere with work duties or violate other company policies.',
                'General Use'),
            PolicyClause('AU-1.2', 'Users shall not install unauthorised software '
                'on company devices. All software requests must go through the '
                'IT Service Desk.',
                'General Use'),
            PolicyClause('AU-2.1', 'Company email shall not be used for personal '
                'business, political campaigning, or distribution of non-business '
                'content to large groups.',
                'Email'),
            PolicyClause('AU-2.2', 'Sensitive data shall not be transmitted via '
                'email without encryption. The approved email encryption tool '
                'shall be used for all confidential communications.',
                'Email'),
            PolicyClause('AU-3.1', 'Remote access to company systems shall be '
                'conducted through the approved VPN solution. Split tunnelling '
                'is not permitted.',
                'Remote Access'),
        ]),

    'sla_policy': PolicyDocument(
        'Service Level Agreement Policy', 'sla_policy', 'v1',
        [
            PolicyClause('SL-1.1', 'The platform shall maintain 99.9%% uptime '
                'measured monthly, excluding scheduled maintenance windows.',
                'Availability'),
            PolicyClause('SL-1.2', 'Scheduled maintenance windows shall be '
                'communicated at least 7 days in advance and limited to '
                'Sunday 02:00-06:00 UTC.',
                'Availability'),
            PolicyClause('SL-2.1', 'P1 incidents (service down) shall receive '
                'initial response within 30 minutes and resolution within 4 hours.',
                'Incident Response'),
            PolicyClause('SL-2.2', 'P2 incidents (major degradation) shall receive '
                'initial response within 1 hour and resolution within 8 hours.',
                'Incident Response'),
            PolicyClause('SL-3.1', 'Service credits shall be issued for SLA '
                'breaches: 10%% credit for uptime below 99.9%%, 25%% credit '
                'for uptime below 99.5%%.',
                'Credits'),
        ]),

    'hr_policy': PolicyDocument(
        'Remote Work Policy', 'hr_policy', 'v1',
        [
            PolicyClause('HR-1.1', 'Employees may work remotely up to 2 days per week '
                'with manager approval. Remote work days must be agreed in advance '
                'and documented in the HR system.',
                'Eligibility'),
            PolicyClause('HR-1.2', 'Remote work is available to employees who have '
                'completed their probation period (6 months) and have a satisfactory '
                'performance rating.',
                'Eligibility'),
            PolicyClause('HR-2.1', 'Remote workers shall be available during core '
                'hours (10:00-16:00 local time) and respond to communications '
                'within 1 hour.',
                'Expectations'),
            PolicyClause('HR-2.2', 'Employees are responsible for maintaining a '
                'suitable home workspace with reliable internet connectivity. '
                'The company does not provide home office equipment.',
                'Expectations'),
            PolicyClause('HR-3.1', 'Remote work arrangements may be revoked with '
                '2 weeks notice if performance standards are not maintained.',
                'Compliance'),
        ]),

    'vendor_management': PolicyDocument(
        'Vendor Management Policy', 'vendor_management', 'v1',
        [
            PolicyClause('VM-1.1', 'All vendors processing company data shall '
                'undergo a security assessment before contract signing. The '
                'assessment shall include a questionnaire and evidence review.',
                'Assessment'),
            PolicyClause('VM-1.2', 'Vendor risk is classified as High, Medium, '
                'or Low based on data sensitivity and access level. High-risk '
                'vendors require annual reassessment.',
                'Assessment'),
            PolicyClause('VM-2.1', 'Vendor contracts shall include data processing '
                'terms, confidentiality clauses, and right-to-audit provisions.',
                'Contracts'),
            PolicyClause('VM-2.2', 'Contract renewals for high-risk vendors '
                'require sign-off from both the business owner and the '
                'Chief Information Security Officer.',
                'Contracts'),
        ]),
}

print(f'Loaded {len(OLD_POLICIES)} policy domains (v1)')
for k, p in OLD_POLICIES.items():
    print(f'  {p.name}: {len(p.clauses)} clauses')

In [ ]:
NEW_POLICIES = {
    'data_security': PolicyDocument(
        'Data Security Policy', 'data_security', 'v2',
        [
            PolicyClause('DS-1.1', 'All data at rest shall be encrypted using AES-256 '
                'with customer-managed keys (CMK). Encryption keys are managed '
                'centrally via AWS KMS and rotated every 90 days.',
                'Encryption'),
            PolicyClause('DS-1.2', 'Data in transit shall be protected using TLS 1.3. '
                'TLS 1.2 is no longer permitted for new deployments. Legacy systems '
                'must upgrade by Q4 2026.',
                'Encryption'),
            PolicyClause('DS-1.3', 'Database-level encryption using Transparent Data '
                'Encryption (TDE) is mandatory for all database instances containing '
                'Confidential or Restricted data.',
                'Encryption'),
            PolicyClause('DS-2.1', 'Access to production systems requires manager approval '
                'and is reviewed monthly. All access follows the principle of least '
                'privilege with just-in-time (JIT) elevation for privileged operations.',
                'Access Control'),
            PolicyClause('DS-2.2', 'Multi-factor authentication is mandatory for all '
                'user accounts. Acceptable MFA methods include TOTP authenticator apps '
                'and FIDO2 hardware keys. SMS-based MFA is no longer permitted.',
                'Access Control'),
            PolicyClause('DS-3.1', 'Vulnerability scans shall be conducted weekly '
                'on all internet-facing systems and monthly on internal systems. '
                'Critical vulnerabilities shall be remediated within 72 hours. '
                'High vulnerabilities within 14 days.',
                'Vulnerability Management'),
            PolicyClause('DS-3.2', 'External penetration testing shall be performed '
                'semi-annually by a CREST-certified provider. Red team exercises '
                'shall be conducted annually.',
                'Vulnerability Management'),
            PolicyClause('DS-4.1', 'Security incidents shall be reported to the '
                'Security Operations Centre (SOC) within 1 hour of detection. '
                'A post-incident review shall be completed within 5 business days. '
                'Regulatory notification shall be made within 72 hours where required.',
                'Incident Response'),
        ]),

    'data_governance': PolicyDocument(
        'Data Governance Policy', 'data_governance', 'v2',
        [
            PolicyClause('DG-1.1', 'All data assets shall be classified into four '
                'categories: Public, Internal, Confidential, and Restricted. '
                'Classification is the responsibility of the data owner and must '
                'be reviewed annually.',
                'Data Classification'),
            PolicyClause('DG-1.2', 'Data owners shall be identified for all data '
                'assets, not only critical ones. Ownership is assigned at the '
                'department head level with a designated data steward for '
                'day-to-day management.',
                'Data Classification'),
            PolicyClause('DG-1.3', 'A data catalogue shall be maintained listing all '
                'data assets, their classification, owner, steward, and processing '
                'purpose. The catalogue shall be updated within 5 business days of '
                'any change.',
                'Data Classification'),
            PolicyClause('DG-2.1', 'Personal data shall be retained for no longer '
                'than necessary for the purpose it was collected. Default retention '
                'period is 2 years, reduced from the previous 3-year default.',
                'Retention'),
            PolicyClause('DG-2.2', 'Data deletion requests shall be processed within '
                '15 business days. Automated confirmation of deletion shall be '
                'provided to the requesting party, including a certificate of '
                'destruction.',
                'Retention'),
            PolicyClause('DG-3.1', 'Cross-border data transfers require approval from '
                'the Data Protection Officer (DPO). Transfers to non-adequate '
                'countries require Standard Contractual Clauses and a Transfer '
                'Impact Assessment.',
                'Cross-Border'),
            PolicyClause('DG-3.2', 'All cross-border transfers shall be logged in '
                'the data transfer register with purpose, legal basis, and '
                'recipient details.',
                'Cross-Border'),
        ]),

    'acceptable_use': PolicyDocument(
        'Acceptable Use Policy', 'acceptable_use', 'v2',
        [
            PolicyClause('AU-1.1', 'Company IT resources are provided for business '
                'purposes. Limited personal use is permitted provided it does not '
                'interfere with work duties or violate other company policies.',
                'General Use'),
            PolicyClause('AU-1.2', 'Users shall not install unauthorised software '
                'on company devices. All software requests must go through the '
                'IT Service Desk. Use of generative AI tools requires prior '
                'approval from the Data Governance team.',
                'General Use'),
            PolicyClause('AU-1.3', 'Company data shall not be entered into public '
                'generative AI services (e.g., public ChatGPT, Claude, Gemini) '
                'unless the service has been approved and a data processing '
                'agreement is in place.',
                'General Use'),
            PolicyClause('AU-2.1', 'Company email shall not be used for personal '
                'business, political campaigning, or distribution of non-business '
                'content to large groups.',
                'Email'),
            PolicyClause('AU-2.2', 'Sensitive data shall not be transmitted via email '
                'without encryption. The approved email encryption tool shall be '
                'used for all confidential communications. Auto-classification '
                'labels are mandatory on all outgoing emails.',
                'Email'),
            PolicyClause('AU-3.1', 'Remote access to company systems shall be '
                'conducted through the approved Zero Trust Network Access (ZTNA) '
                'solution. The legacy VPN will be decommissioned by Q2 2026.',
                'Remote Access'),
        ]),

    'sla_policy': PolicyDocument(
        'Service Level Agreement Policy', 'sla_policy', 'v2',
        [
            PolicyClause('SL-1.1', 'The platform shall maintain 99.95%% uptime '
                'measured monthly, excluding scheduled maintenance windows. '
                'Uptime is measured using synthetic monitoring from 5 global regions.',
                'Availability'),
            PolicyClause('SL-1.2', 'Scheduled maintenance windows shall be '
                'communicated at least 14 days in advance. Maintenance shall be '
                'limited to Sunday 02:00-04:00 UTC with zero-downtime deployment '
                'preferred.',
                'Availability'),
            PolicyClause('SL-2.1', 'P1 incidents (service down) shall receive '
                'initial response within 15 minutes and resolution within 1 hour. '
                'Automated escalation to VP Engineering after 30 minutes.',
                'Incident Response'),
            PolicyClause('SL-2.2', 'P2 incidents (major degradation) shall receive '
                'initial response within 30 minutes and resolution within 4 hours.',
                'Incident Response'),
            PolicyClause('SL-2.3', 'All P1 and P2 incidents shall have a blameless '
                'post-mortem published within 3 business days, including root cause, '
                'timeline, and corrective actions.',
                'Incident Response'),
            PolicyClause('SL-3.1', 'Service credits shall be issued automatically '
                'for SLA breaches: 10%% credit for uptime below 99.95%%, 25%% '
                'credit for uptime below 99.9%%, 50%% credit for uptime below '
                '99.0%%.',
                'Credits'),
        ]),

    'hr_policy': PolicyDocument(
        'Remote Work Policy', 'hr_policy', 'v2',
        [
            PolicyClause('HR-1.1', 'Employees may work remotely up to 3 days per week '
                'with manager approval. A hybrid schedule shall be agreed quarterly '
                'and documented in the HR system.',
                'Eligibility'),
            PolicyClause('HR-1.2', 'Remote work is available to employees who have '
                'completed their probation period (3 months) and have a satisfactory '
                'performance rating.',
                'Eligibility'),
            PolicyClause('HR-1.3', 'Fully remote positions may be approved for roles '
                'where physical presence is not required, subject to VP and HR '
                'Director approval.',
                'Eligibility'),
            PolicyClause('HR-2.1', 'Remote workers shall be available during core '
                'hours (10:00-15:00 local time) and respond to communications '
                'within 30 minutes during core hours.',
                'Expectations'),
            PolicyClause('HR-2.2', 'The company provides a one-time home office '
                'stipend of $500 for ergonomic equipment and reimburses broadband '
                'costs up to $50/month for remote-eligible employees.',
                'Expectations'),
            PolicyClause('HR-3.1', 'Remote work arrangements may be adjusted with '
                '4 weeks notice following a structured performance review process.',
                'Compliance'),
        ]),

    'vendor_management': PolicyDocument(
        'Vendor Management Policy', 'vendor_management', 'v2',
        [
            PolicyClause('VM-1.1', 'All vendors processing company data shall '
                'undergo a comprehensive security assessment before contract '
                'signing. The assessment includes a questionnaire, evidence '
                'review, and SOC 2 report evaluation.',
                'Assessment'),
            PolicyClause('VM-1.2', 'Vendor risk is classified as Critical, High, '
                'Medium, or Low based on data sensitivity, access level, and '
                'business criticality. Critical and high-risk vendors require '
                'semi-annual reassessment.',
                'Assessment'),
            PolicyClause('VM-1.3', 'AI and machine learning vendors require '
                'additional assessment of model training data practices, '
                'output ownership, and data retention policies.',
                'Assessment'),
            PolicyClause('VM-2.1', 'Vendor contracts shall include data processing '
                'terms, confidentiality clauses, right-to-audit provisions, '
                'and incident notification requirements with a maximum '
                '24-hour notification window.',
                'Contracts'),
            PolicyClause('VM-2.2', 'Contract renewals for critical and high-risk '
                'vendors require sign-off from the business owner, CISO, and '
                'the Data Protection Officer.',
                'Contracts'),
            PolicyClause('VM-3.1', 'A vendor register shall be maintained with '
                'risk classification, assessment dates, contract terms, and '
                'data processing details. The register is reviewed quarterly '
                'by the Vendor Governance Committee.',
                'Oversight'),
        ]),
}

print(f'\nLoaded {len(NEW_POLICIES)} policy domains (v2)')
for k, p in NEW_POLICIES.items():
    old_n = len(OLD_POLICIES[k].clauses)
    new_n = len(p.clauses)
    print(f'  {p.name}: v1={old_n} clauses -> v2={new_n} clauses (delta={new_n - old_n:+d})')

## Part 3: Clause-Level Diff Engine

The diff engine aligns old and new clauses using semantic similarity,
then classifies each pair as UNCHANGED, MODIFIED, ADDED, or DELETED.

In [ ]:
class ClauseEmbedder:
    def __init__(self, dim=64):
        self.dim, self.vocab, self.idf, self.proj = dim, {}, {}, None

    def fit(self, texts):
        df = Counter()
        for t in texts:
            for tok in set(re.findall(r'[a-z0-9]+', t.lower())):
                df[tok] += 1
        self.vocab = {t: i for i, t in enumerate(sorted(df))}
        n = len(texts)
        self.idf = {t: math.log(n / (1 + c)) for t, c in df.items()}
        np.random.seed(42)
        self.proj = np.random.randn(len(self.vocab), self.dim) / math.sqrt(self.dim)

    def encode(self, text):
        vec = np.zeros(len(self.vocab))
        for tok, cnt in Counter(re.findall(r'[a-z0-9]+', text.lower())).items():
            if tok in self.vocab:
                vec[self.vocab[tok]] = cnt * self.idf.get(tok, 1.0)
        p = vec @ self.proj
        norm = np.linalg.norm(p)
        return p / norm if norm > 0 else p


# Fit on all clauses from both versions
all_clause_texts = []
for d in list(OLD_POLICIES.values()) + list(NEW_POLICIES.values()):
    all_clause_texts.extend(c.text for c in d.clauses)

clause_embedder = ClauseEmbedder(64)
clause_embedder.fit(all_clause_texts)
print(f'Embedder fit on {len(all_clause_texts)} clauses, vocab={len(clause_embedder.vocab)}')

In [ ]:
@dataclass
class ClauseDiff:
    old_clause: Optional[PolicyClause]
    new_clause: Optional[PolicyClause]
    status: str             # UNCHANGED | MODIFIED | ADDED | DELETED
    similarity: float
    word_diff: str          # human-readable inline diff
    section: str


def compute_clause_diff(old_doc, new_doc, embedder,
                        unchanged_threshold=0.95, match_threshold=0.60):
    # Encode all clauses
    old_embs = [embedder.encode(c.text) for c in old_doc.clauses]
    new_embs = [embedder.encode(c.text) for c in new_doc.clauses]

    # Pairwise similarity matrix
    sim_matrix = np.zeros((len(old_embs), len(new_embs)))
    for i, oe in enumerate(old_embs):
        for j, ne in enumerate(new_embs):
            sim_matrix[i, j] = float(oe @ ne)

    # Greedy best-match alignment
    used_old = set()
    used_new = set()
    diffs = []

    # Sort all pairs by similarity descending
    pairs = []
    for i in range(len(old_embs)):
        for j in range(len(new_embs)):
            pairs.append((sim_matrix[i, j], i, j))
    pairs.sort(key=lambda x: -x[0])

    for sim, i, j in pairs:
        if i in used_old or j in used_new:
            continue
        if sim < match_threshold:
            break
        used_old.add(i)
        used_new.add(j)

        old_c = old_doc.clauses[i]
        new_c = new_doc.clauses[j]

        if sim >= unchanged_threshold:
            status = 'UNCHANGED'
            word_diff = '(no changes)'
        else:
            status = 'MODIFIED'
            # Generate inline word diff
            old_words = old_c.text.split()
            new_words = new_c.text.split()
            d = difflib.unified_diff(old_words, new_words, n=0, lineterm='')
            diff_lines = [l for l in d if l.startswith('+') or l.startswith('-')]
            diff_lines = [l for l in diff_lines if not l.startswith('+++') and not l.startswith('---')]
            word_diff = ' '.join(diff_lines[:20]) if diff_lines else '(minor changes)'

        diffs.append(ClauseDiff(
            old_clause=old_c, new_clause=new_c,
            status=status, similarity=round(sim, 4),
            word_diff=word_diff, section=new_c.section,
        ))

    # Unmatched old clauses = DELETED
    for i in range(len(old_doc.clauses)):
        if i not in used_old:
            c = old_doc.clauses[i]
            diffs.append(ClauseDiff(
                old_clause=c, new_clause=None,
                status='DELETED', similarity=0.0,
                word_diff=f'REMOVED: {c.text[:80]}...',
                section=c.section,
            ))

    # Unmatched new clauses = ADDED
    for j in range(len(new_doc.clauses)):
        if j not in used_new:
            c = new_doc.clauses[j]
            diffs.append(ClauseDiff(
                old_clause=None, new_clause=c,
                status='ADDED', similarity=0.0,
                word_diff=f'NEW: {c.text[:80]}...',
                section=c.section,
            ))

    return diffs


# Test on data security policy
ds_diffs = compute_clause_diff(
    OLD_POLICIES['data_security'], NEW_POLICIES['data_security'], clause_embedder
)
print(f'Data Security Policy: {len(ds_diffs)} clause-level diffs')
for d in ds_diffs:
    old_id = d.old_clause.clause_id if d.old_clause else '---'
    new_id = d.new_clause.clause_id if d.new_clause else '---'
    print(f'  {old_id} -> {new_id}: {d.status} (sim={d.similarity:.3f})')

## Part 4: Change Classifier

Classifies each diff as **substantive**, **editorial**, or **structural**
based on the nature of the change.

In [ ]:
SUBSTANTIVE_SIGNALS = [
    r'\d+\s*(%|percent|days?|hours?|minutes?|weeks?|months?|years?)',
    r'\$\d+', r'mandatory|required|prohibited|shall not|must not',
    r'shall|must|will', r'no longer permitted|decommission|sunset',
    r'added|removed|replaced|increased|decreased|reduced|expanded',
    r'new\b|additional|furthermore', r'critical|high-risk',
]

EDITORIAL_SIGNALS = [
    r'shall\b.*will\b|will\b.*shall\b',  # shall <-> will
    r'\b(the|a|an)\b',  # article changes
]


@dataclass
class ClassifiedChange:
    diff: ClauseDiff
    change_type: str        # substantive | editorial | structural
    impact_level: str       # HIGH | MEDIUM | LOW | NONE
    change_summary: str     # one-line summary of what changed
    affected_functions: list # business functions affected


FUNCTION_KEYWORDS = {
    'IT Security': ['encrypt', 'mfa', 'authentication', 'vulnerability',
                    'penetration', 'firewall', 'access control', 'incident',
                    'soc', 'security'],
    'Legal & Compliance': ['gdpr', 'regulatory', 'compliance', 'contract',
                           'legal', 'data protection', 'cross-border',
                           'audit', 'certification'],
    'HR & People': ['employee', 'remote work', 'probation', 'manager',
                    'performance', 'stipend', 'training', 'onboarding'],
    'Engineering': ['deployment', 'uptime', 'sla', 'incident response',
                    'maintenance', 'api', 'system', 'platform'],
    'Data Management': ['data', 'retention', 'deletion', 'classification',
                        'catalogue', 'owner', 'steward'],
    'Procurement': ['vendor', 'contract', 'assessment', 'renewal',
                    'sign-off', 'register'],
}


def classify_change(diff):
    if diff.status == 'UNCHANGED':
        return ClassifiedChange(
            diff=diff, change_type='none', impact_level='NONE',
            change_summary='No change.', affected_functions=[],
        )

    # Get the relevant text for analysis
    old_text = diff.old_clause.text if diff.old_clause else ''
    new_text = diff.new_clause.text if diff.new_clause else ''
    combined = old_text + ' ' + new_text

    # Detect affected business functions
    affected = []
    combined_lower = combined.lower()
    for func, kws in FUNCTION_KEYWORDS.items():
        if any(kw in combined_lower for kw in kws):
            affected.append(func)

    # Classify change type
    if diff.status == 'ADDED':
        change_type = 'substantive'
        impact_level = 'HIGH'
        change_summary = f'New clause added: {new_text[:100]}'
    elif diff.status == 'DELETED':
        change_type = 'substantive'
        impact_level = 'HIGH'
        change_summary = f'Clause removed: {old_text[:100]}'
    else:
        # MODIFIED -- check if substantive or editorial
        # Find actual differences
        old_words = set(re.findall(r'[a-z0-9]+', old_text.lower()))
        new_words = set(re.findall(r'[a-z0-9]+', new_text.lower()))
        added_words = new_words - old_words
        removed_words = old_words - new_words
        diff_text = ' '.join(added_words | removed_words)

        # Check for substantive signals
        subst_hits = sum(1 for pat in SUBSTANTIVE_SIGNALS
                         if re.search(pat, diff_text + ' ' + new_text, re.I))

        if subst_hits >= 2 or diff.similarity < 0.80:
            change_type = 'substantive'
            impact_level = 'HIGH' if diff.similarity < 0.75 else 'MEDIUM'
        elif len(added_words | removed_words) > 15:
            change_type = 'substantive'
            impact_level = 'MEDIUM'
        else:
            change_type = 'editorial'
            impact_level = 'LOW'

        # Build change summary by finding key differences
        key_adds = sorted(added_words - {'the','a','an','and','or','is','are','to','of','in','for'})[:5]
        key_removes = sorted(removed_words - {'the','a','an','and','or','is','are','to','of','in','for'})[:5]
        parts = []
        if key_removes:
            parts.append(f'Removed: {", ".join(key_removes)}')
        if key_adds:
            parts.append(f'Added: {", ".join(key_adds)}')
        change_summary = '; '.join(parts) if parts else 'Minor wording change'

    return ClassifiedChange(
        diff=diff, change_type=change_type, impact_level=impact_level,
        change_summary=change_summary, affected_functions=affected,
    )


# Classify all data security diffs
ds_classified = [classify_change(d) for d in ds_diffs]
print('Data Security Policy -- Classified Changes:')
for c in ds_classified:
    if c.change_type == 'none':
        continue
    old_id = c.diff.old_clause.clause_id if c.diff.old_clause else 'NEW'
    new_id = c.diff.new_clause.clause_id if c.diff.new_clause else 'DEL'
    print(f'  {old_id}->{new_id}: {c.diff.status} [{c.change_type}/{c.impact_level}]')
    print(f'    {c.change_summary[:100]}')
    print(f'    Affects: {c.affected_functions}')

## Part 5: Structured Summarisation

### What Is Structured Summarisation?

Traditional summarisation produces prose. **Structured summarisation**
produces a typed report with predefined fields:

```json
{
  "domain": "Data Security",
  "total_changes": 8,
  "substantive_changes": 5,
  "impact_summary": "Significant tightening...",
  "action_items": [...],
  "affected_teams": [...]
}
```

This is more useful than prose because downstream systems (dashboards,
ticketing, compliance tools) can consume the structured output directly.

### Summarisation Strategy

| Step | What | How |
|---|---|---|
| 1 | Group changes by section | Cluster related diffs |
| 2 | Identify themes | Count substantive vs editorial per section |
| 3 | Generate action items | Each substantive change -> action |
| 4 | Map stakeholders | Change type -> affected business function |
| 5 | Assess overall impact | Aggregate section impacts |

In [ ]:
@dataclass
class ImpactSummary:
    domain: str
    policy_name: str
    total_changes: int
    substantive_count: int
    editorial_count: int
    added_count: int
    deleted_count: int
    overall_impact: str          # HIGH | MEDIUM | LOW
    executive_summary: str
    section_summaries: dict
    action_items: list
    affected_teams: list
    risk_notes: list


def summarise_policy_changes(domain, classified_changes):
    # Filter out unchanged
    changes = [c for c in classified_changes if c.change_type != 'none']
    substantive = [c for c in changes if c.change_type == 'substantive']
    editorial = [c for c in changes if c.change_type == 'editorial']
    added = [c for c in changes if c.diff.status == 'ADDED']
    deleted = [c for c in changes if c.diff.status == 'DELETED']

    # Overall impact
    high_count = sum(1 for c in changes if c.impact_level == 'HIGH')
    if high_count >= 3:
        overall = 'HIGH'
    elif high_count >= 1 or len(substantive) >= 3:
        overall = 'MEDIUM'
    else:
        overall = 'LOW'

    # Section summaries
    section_groups = defaultdict(list)
    for c in changes:
        section_groups[c.diff.section].append(c)

    section_sums = {}
    for section, sec_changes in section_groups.items():
        sec_sub = sum(1 for c in sec_changes if c.change_type == 'substantive')
        sec_adds = sum(1 for c in sec_changes if c.diff.status == 'ADDED')
        section_sums[section] = {
            'changes': len(sec_changes),
            'substantive': sec_sub,
            'additions': sec_adds,
            'key_change': sec_changes[0].change_summary[:120] if sec_changes else '',
        }

    # Action items from substantive changes
    actions = []
    for c in substantive:
        if c.diff.status == 'ADDED':
            new_text = c.diff.new_clause.text[:80] if c.diff.new_clause else ''
            actions.append(f'IMPLEMENT new requirement: {new_text}')
        elif c.diff.status == 'DELETED':
            old_text = c.diff.old_clause.text[:80] if c.diff.old_clause else ''
            actions.append(f'RETIRE process for: {old_text}')
        elif c.diff.status == 'MODIFIED':
            actions.append(f'UPDATE process: {c.change_summary[:100]}')

    # Affected teams
    all_funcs = set()
    for c in changes:
        all_funcs.update(c.affected_functions)

    # Risk notes
    risks = []
    if len(added) >= 3:
        risks.append(f'{len(added)} new clauses added -- significant new obligations')
    if deleted:
        risks.append(f'{len(deleted)} clause(s) removed -- verify no compliance gap')
    if high_count >= 2:
        risks.append(f'{high_count} high-impact changes -- requires management review')

    # Executive summary (simulated LLM output)
    name = list(OLD_POLICIES.values())[list(OLD_POLICIES.keys()).index(domain)].name
    exec_parts = []
    if overall == 'HIGH':
        exec_parts.append(f'The {name} has undergone significant revision')
    else:
        exec_parts.append(f'The {name} has been updated')
    exec_parts.append(f'with {len(changes)} changes ({len(substantive)} substantive)')
    if added:
        exec_parts.append(f'including {len(added)} new clause(s)')
    exec_parts.append(f'affecting {len(all_funcs)} business function(s).')
    if actions:
        exec_parts.append(f'{len(actions)} action item(s) identified.')

    return ImpactSummary(
        domain=domain, policy_name=name,
        total_changes=len(changes),
        substantive_count=len(substantive),
        editorial_count=len(editorial),
        added_count=len(added),
        deleted_count=len(deleted),
        overall_impact=overall,
        executive_summary=' '.join(exec_parts),
        section_summaries=dict(section_sums),
        action_items=actions,
        affected_teams=sorted(all_funcs),
        risk_notes=risks,
    )


print('Summariser ready')

## Part 6: End-to-End Pipeline

Process all 6 policy domains through the full pipeline:

```
old_doc + new_doc  ->  DIFF  ->  CLASSIFY  ->  SUMMARISE  ->  REPORT
```

In [ ]:
class PolicyDiffPipeline:
    def __init__(self, embedder):
        self.embedder = embedder

    def process(self, old_doc, new_doc):
        diffs = compute_clause_diff(old_doc, new_doc, self.embedder)
        classified = [classify_change(d) for d in diffs]
        summary = summarise_policy_changes(old_doc.domain, classified)
        return {
            'diffs': diffs,
            'classified': classified,
            'summary': summary,
        }

    def process_all(self, old_policies, new_policies):
        results = {}
        for domain in old_policies:
            if domain in new_policies:
                results[domain] = self.process(
                    old_policies[domain], new_policies[domain]
                )
        return results


pipeline = PolicyDiffPipeline(clause_embedder)
all_results = pipeline.process_all(OLD_POLICIES, NEW_POLICIES)

print('POLICY DIFF PIPELINE -- EXECUTIVE DASHBOARD')
print('=' * 75)
for domain, r in all_results.items():
    s = r['summary']
    impact_color = {'HIGH': '!!!', 'MEDIUM': ' ! ', 'LOW': '   '}[s.overall_impact]
    print(f'\n[{impact_color}] {s.policy_name} (v1 -> v2)')
    print(f'    Impact: {s.overall_impact} | Changes: {s.total_changes} '
          f'(substantive: {s.substantive_count}, editorial: {s.editorial_count})')
    print(f'    Added: {s.added_count} | Deleted: {s.deleted_count}')
    print(f'    Affected: {", ".join(s.affected_teams)}')
    print(f'    Summary: {s.executive_summary[:120]}')

In [ ]:
# Detailed view: one policy
domain = 'data_security'
r = all_results[domain]
s = r['summary']

print(f'DETAILED REPORT: {s.policy_name}')
print('=' * 75)
print(f'\nExecutive Summary:')
print(f'  {s.executive_summary}')

print(f'\nSection-Level Changes:')
for section, info in s.section_summaries.items():
    print(f'  {section}: {info["changes"]} changes ({info["substantive"]} substantive)')
    print(f'    Key: {info["key_change"][:100]}')

print(f'\nAction Items ({len(s.action_items)}):')
for i, action in enumerate(s.action_items, 1):
    print(f'  {i}. {action[:100]}')

print(f'\nRisk Notes:')
for r_note in s.risk_notes:
    print(f'  - {r_note}')

## Part 7: Evaluation

We measure the pipeline across three quality axes:

| Metric | What | Target |
|---|---|---|
| Alignment accuracy | Did we match the right clauses? | >90%% |
| Classification precision | Substantive vs editorial vs structural | >85%% |
| Coverage completeness | Were all changes detected? | 100%% |

In [ ]:
# Aggregate metrics across all policies
eval_rows = []
for domain, r in all_results.items():
    s = r['summary']
    classified = r['classified']
    diffs = r['diffs']

    total_old = len(OLD_POLICIES[domain].clauses)
    total_new = len(NEW_POLICIES[domain].clauses)
    total_diffs = len(diffs)

    # Alignment -- how many old/new clauses were matched
    matched = sum(1 for d in diffs if d.status in ('UNCHANGED', 'MODIFIED'))
    unmatched_old = sum(1 for d in diffs if d.status == 'DELETED')
    unmatched_new = sum(1 for d in diffs if d.status == 'ADDED')
    coverage = total_diffs / max(total_old, total_new)

    # Change distribution
    status_counts = Counter(d.status for d in diffs)
    type_counts = Counter(c.change_type for c in classified)

    eval_rows.append({
        'domain': domain,
        'policy': s.policy_name,
        'old_clauses': total_old,
        'new_clauses': total_new,
        'matched': matched,
        'added': unmatched_new,
        'deleted': unmatched_old,
        'unchanged': status_counts.get('UNCHANGED', 0),
        'modified': status_counts.get('MODIFIED', 0),
        'substantive': type_counts.get('substantive', 0),
        'editorial': type_counts.get('editorial', 0),
        'overall_impact': s.overall_impact,
        'action_items': len(s.action_items),
        'coverage': round(coverage, 3),
    })

eval_df = pd.DataFrame(eval_rows)
print('EVALUATION SUMMARY')
print('=' * 100)
print(eval_df[['policy', 'old_clauses', 'new_clauses', 'matched', 'added',
               'modified', 'substantive', 'overall_impact', 'action_items']].to_string(index=False))

## Part 8: Visualisations

In [ ]:
STATUS_COLORS = {'UNCHANGED': '#95a5a6', 'MODIFIED': '#f39c12',
                 'ADDED': '#27ae60', 'DELETED': '#e74c3c'}
IMPACT_COLORS = {'HIGH': '#e74c3c', 'MEDIUM': '#f39c12', 'LOW': '#27ae60'}

# 1  Change status distribution by policy
status_rows = []
for domain, r in all_results.items():
    for d in r['diffs']:
        status_rows.append({
            'policy': r['summary'].policy_name,
            'status': d.status,
        })
status_df = pd.DataFrame(status_rows)

fig = px.histogram(status_df, x='policy', color='status', barmode='stack',
                   color_discrete_map=STATUS_COLORS,
                   title='Clause-Level Change Status by Policy',
                   category_orders={'status': ['UNCHANGED', 'MODIFIED', 'ADDED', 'DELETED']},
                   template='plotly_white', height=430)
fig.update_xaxes(tickangle=-30)
fig.show()

In [ ]:
# 2  Impact level summary
impact_data = eval_df[['policy', 'overall_impact']].copy()
impact_data['impact_score'] = impact_data['overall_impact'].map({'HIGH': 3, 'MEDIUM': 2, 'LOW': 1})
impact_data = impact_data.sort_values('impact_score', ascending=True)

fig = go.Figure(go.Bar(
    x=impact_data['impact_score'],
    y=impact_data['policy'],
    orientation='h',
    marker_color=[IMPACT_COLORS[i] for i in impact_data['overall_impact']],
    text=impact_data['overall_impact'],
    textposition='inside',
))
fig.update_layout(
    title='Overall Impact Level by Policy',
    xaxis_title='Impact Score', xaxis=dict(tickvals=[1,2,3], ticktext=['LOW','MEDIUM','HIGH']),
    template='plotly_white', height=380, showlegend=False,
)
fig.show()

In [ ]:
# 3  Radar: change profile per policy
radar_dims = ['matched', 'modified', 'added', 'substantive', 'action_items']
# Normalise to 0-1 range
radar_norm = eval_df[radar_dims].copy()
for col in radar_dims:
    mx = radar_norm[col].max()
    if mx > 0:
        radar_norm[col] = radar_norm[col] / mx

DOMAIN_COLORS = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#e67e22', '#1abc9c']
fig = go.Figure()
for i, (_, row) in enumerate(eval_df.iterrows()):
    vals = [radar_norm.iloc[i][d] for d in radar_dims]
    fig.add_trace(go.Scatterpolar(
        r=vals + [vals[0]], theta=radar_dims + [radar_dims[0]],
        name=row['policy'], fill='toself',
        line_color=DOMAIN_COLORS[i % len(DOMAIN_COLORS)], opacity=0.45,
    ))
fig.update_layout(
    polar=dict(radialaxis=dict(range=[0, 1], visible=True)),
    title='Change Profile Radar (normalised)',
    template='plotly_white', height=520,
)
fig.show()

In [ ]:
# 4  Affected teams heatmap
team_rows = []
for domain, r in all_results.items():
    team_counter = Counter()
    for c in r['classified']:
        if c.change_type != 'none':
            for t in c.affected_functions:
                team_counter[t] += 1
    for team, count in team_counter.items():
        team_rows.append({'policy': r['summary'].policy_name,
                          'team': team, 'changes': count})

team_df = pd.DataFrame(team_rows)
if not team_df.empty:
    team_pivot = team_df.pivot_table(index='team', columns='policy',
                                     values='changes', fill_value=0)
    fig = px.imshow(team_pivot.values,
                    x=list(team_pivot.columns), y=list(team_pivot.index),
                    color_continuous_scale='YlOrRd', text_auto=True,
                    title='Changes Affecting Each Business Function',
                    labels=dict(x='Policy', y='Business Function', color='Changes'))
    fig.update_layout(template='plotly_white', height=420)
    fig.show()

In [ ]:
# 5  Action items by policy
action_data = eval_df[['policy', 'action_items', 'overall_impact']].sort_values(
    'action_items', ascending=True)

fig = go.Figure(go.Bar(
    x=action_data['action_items'],
    y=action_data['policy'],
    orientation='h',
    marker_color=[IMPACT_COLORS[i] for i in action_data['overall_impact']],
    text=action_data['action_items'],
    textposition='outside',
))
fig.update_layout(
    title='Action Items Generated per Policy',
    xaxis_title='Number of Action Items',
    template='plotly_white', height=380, showlegend=False,
)
fig.show()

In [ ]:
# 6  Similarity distribution for matched clauses
sim_rows = []
for domain, r in all_results.items():
    for d in r['diffs']:
        if d.status in ('MODIFIED', 'UNCHANGED'):
            sim_rows.append({
                'policy': r['summary'].policy_name,
                'similarity': d.similarity,
                'status': d.status,
            })

sim_df = pd.DataFrame(sim_rows)
if not sim_df.empty:
    fig = px.strip(sim_df, x='policy', y='similarity', color='status',
                   color_discrete_map=STATUS_COLORS,
                   title='Clause Similarity Scores (Matched Pairs)',
                   labels={'similarity': 'Cosine Similarity'},
                   template='plotly_white', height=400)
    fig.add_hline(y=0.95, line_dash='dash', line_color='grey',
                  annotation_text='Unchanged threshold')
    fig.add_hline(y=0.60, line_dash='dash', line_color='red',
                  annotation_text='Match threshold')
    fig.show()

In [ ]:
# 7  Full change log table
log_rows = []
for domain, r in all_results.items():
    for c in r['classified']:
        if c.change_type == 'none':
            continue
        old_id = c.diff.old_clause.clause_id if c.diff.old_clause else '---'
        new_id = c.diff.new_clause.clause_id if c.diff.new_clause else '---'
        log_rows.append({
            'Policy': r['summary'].policy_name,
            'Old ID': old_id,
            'New ID': new_id,
            'Status': c.diff.status,
            'Type': c.change_type.upper(),
            'Impact': c.impact_level,
            'Section': c.diff.section,
            'Summary': c.change_summary[:80],
        })

log_df = pd.DataFrame(log_rows)
print(f'COMPLETE CHANGE LOG: {len(log_df)} changes across {len(all_results)} policies')
print('=' * 120)
log_df

## Part 9: Production Patterns

### Real-World Architecture

```
  ┌─────────────┐     ┌──────────────┐     ┌──────────────┐
  │ Doc Ingestion│     │ Clause       │     │ Semantic     │
  │ PDF / DOCX   │────>│ Segmenter    │────>│ Diff Engine  │
  │ (Unstructured│     │ (heading +   │     │ (embedding + │
  │  .io)        │     │  structure)  │     │  alignment)  │
  └─────────────┘     └──────────────┘     └──────┬───────┘
                                                   │
                                                   ▼
  ┌─────────────────────────────────────────────────────────────┐
  │                    LLM Analysis Layer                       │
  │  ┌──────────────┐  ┌──────────────┐  ┌──────────────────┐ │
  │  │ Change       │  │ Impact       │  │ Action Item      │ │
  │  │ Classifier   │  │ Summariser   │  │ Generator        │ │
  │  │ (subst/edit) │  │ (per-team)   │  │ (Jira tickets)   │ │
  │  └──────────────┘  └──────────────┘  └──────────────────┘ │
  └─────────────────────────────┬───────────────────────────────┘
                                │
                                ▼
  ┌──────────────────────────────────────────────────┐
  │  Stakeholder Dashboard (React / Streamlit)       │
  │  - Change timeline    - Impact heatmap           │
  │  - Action item board  - Compliance audit trail   │
  └──────────────────────────────────────────────────┘
```

### Key Production Choices

| Component | Recommendation |
|---|---|
| **Document parsing** | Unstructured.io or LlamaParse for PDF/DOCX with heading detection |
| **Clause segmentation** | Heading hierarchy + regex section IDs + sentence-boundary fallback |
| **Embeddings** | `BAAI/bge-large-en-v1.5` or OpenAI `text-embedding-3-large` |
| **Alignment** | Hungarian algorithm (optimal) or greedy best-match (fast) |
| **Classification LLM** | GPT-4o / Claude / Qwen3 with structured output (JSON mode) |
| **Summarisation** | Map-reduce: summarise per section, then aggregate |
| **Action items** | LLM generates Jira-compatible tickets with assignee suggestions |
| **Storage** | PostgreSQL with `tsvector` for clause search + pgvector for embeddings |
| **Versioning** | Git-style version DAG with branch/merge for policy variants |

### LangChain Implementation

```python
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import Literal

class ChangeAnalysis(BaseModel):
    change_type: Literal['substantive', 'editorial', 'structural']
    impact_level: Literal['HIGH', 'MEDIUM', 'LOW']
    summary: str
    affected_teams: list[str]
    action_required: str

llm = ChatOllama(model='qwen3', format='json')
parser = JsonOutputParser(pydantic_object=ChangeAnalysis)

prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a policy analyst. Compare the old and new clause text. '
     'Classify the change and assess business impact.\n'
     '{format_instructions}'),
    ('human',
     'OLD CLAUSE: {old_text}\n\n'
     'NEW CLAUSE: {new_text}'),
])

chain = prompt | llm | parser
result = chain.invoke({
    'old_text': old_clause,
    'new_text': new_clause,
    'format_instructions': parser.get_format_instructions(),
})
```

## Summary

### The Policy Diff Pipeline

```
old_doc + new_doc  -->  Clause Diff  -->  Classify  -->  Summarise  -->  Report
```

### What We Built

| Component | Purpose | Output |
|---|---|---|
| **Clause-level differ** | Align old and new clauses via embeddings | UNCHANGED / MODIFIED / ADDED / DELETED |
| **Change classifier** | Distinguish meaning changes from rewording | substantive / editorial / structural |
| **Impact summariser** | Structured summary per business function | Action items, risk notes, stakeholder map |
| **Full pipeline** | Process 6 policy domains end-to-end | Executive dashboard + detailed reports |

### Key Takeaways

| # | Insight |
|---|---|
| 1 | **Semantic diff > line diff** for policy documents -- catches paraphrases, ignores formatting |
| 2 | **Not all changes are equal** -- classifying substantive vs editorial focuses human review |
| 3 | **Structured summarisation** produces actionable output, not just prose |
| 4 | **Impact mapping to business functions** ensures the right stakeholders are notified |
| 5 | **Action items from diffs** close the loop between analysis and execution |
| 6 | **Thresholds need calibration** -- tune match (0.60) and unchanged (0.95) on real data |

### Applicability Beyond Policy Documents

| Domain | Old | New | What Changes |
|---|---|---|---|
| Legal contracts | Previous contract | Renewal draft | Terms, obligations, SLAs |
| Regulatory filings | Current regulation | Proposed amendment | Compliance requirements |
| SOPs | Existing procedure | Updated procedure | Process steps, approvals |
| Product specs | v1.0 spec | v2.0 spec | Features, requirements |

---
*Educational notebook -- NLP / Document Intelligence Portfolio.*